In [23]:
import torch
from pathlib import Path

from inference import inference
from model import LeagueDraftModel
from vocabulary import Vocabulary
from data import load_matches, ChampionDataset
from torch.utils.data import DataLoader
from draft_constraints import mask_logits

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

BATCH_SIZE = 2048

In [19]:
start = Path.cwd().resolve()

PROJECT_ROOT = next(
    path
    for path in (start, *start.parents)
    if (path / '.git').exists()
)

CHECKPOINT_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'checkpoints'
DATA_DIRECTORY = PROJECT_ROOT / 'data' 

In [ ]:
checkpoint = torch.load(CHECKPOINT_DIRECTORY / 'best_model.pth', device, weights_only=True)
state_dict = checkpoint['model_state_dict']
champ_dict = checkpoint['riotid_to_name']

vocab = Vocabulary(champ_dict)
model = LeagueDraftModel(len(vocab), vocab.mask_id)

model.load_state_dict(state_dict)
model.to(device)
model.eval()

encoded_matches = load_matches(DATA_DIRECTORY / 'league_data.db', vocab)

test_data = ChampionDataset(encoded_matches, vocab.mask_id)

In [21]:
eval_loader = DataLoader(
    test_data, 
    batch_size = BATCH_SIZE , 
    shuffle=False,
    pin_memory=(device.type=='cuda')
)

In [ ]:
# calculate top-5 accuracy for entire dataset
top_5_count = 0
for picks, bans, target in eval_loader: 
    with torch.inference_mode():
    
        picks = picks.to(device, non_blocking=(device.type == 'cuda'))
        bans = bans.to(device, non_blocking=(device.type == 'cuda'))
        target = target.to(device, non_blocking=(device.type == 'cuda'))

        logits = model(picks)
        logits = mask_logits(picks, bans, logits)
    
        # returns value indices pairs, only care about index
        _, preds = torch.topk(logits, k=5, dim=1)
    
        top_5_count += torch.sum(torch.sum(preds == target.unsqueeze(1), dim=1))

top_5_accuracy = top_5_count.item() / len(test_data)

In [51]:
print(top_5_accuracy)

0.4075501752150366


In [22]:
game = ['malphite', 'diana', 'ahri', 'masked', 'lulu', 'darius', 'warwick', 'orianna', 'ezreal', 'karma']

print(inference(model, game, vocab, device, k=5))

['Yunara', 'Jinx', 'Aphelios', 'Zeri', 'Tristana']
